# Steam data preprocessing — extended engineered dataset

Extended version of `mle-process-data.ipynb`, using the cleaning and feature definitions from `steam_model_experiments.ipynb`.

**Input:** `data/games.csv`. **Output:** `data/games-extra.parquet` plus a dataset manifest, cleaning report and feature catalog. The existing `games.parquet` is left in place.

Run all cells using the project `.venv` kernel. The notebook is self-contained and does not run another notebook or train a model. It preserves source metadata, the original ownership/game-age/Steam-score fields and normalized list columns, adds every deterministic experiment feature and several additional metadata candidates, then checks the saved Parquet by reading it back.

Ownership columns describe the outcome; they are excluded from model inputs. Reviews, playtime and concurrency remain available for analysis but are excluded from the model-input catalog.

## 1. Configuration and shared ownership definitions

- `Estimated owners (lb)` = lower endpoint of `Estimated owners`.
- `Estimated owners (ub)` = upper endpoint.
- `Estimated owners (mid)` = `(lb + ub) / 2`.

Example: `20000 - 50000` gives **20,000**, **50,000** and **35,000**. All three columns are defined before filtering or target construction using the shared `steam_data.py` helper.

Use the same observation date (**2023-07-05**) and coverage as the experiments: games at least one year old, with positive estimated ownership. Targets are 0 = <20k, 1 = 20k–<100k, 2 = 100k+.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import ast
import hashlib
import html
import importlib.metadata
import json
import os
import platform
import re
import sys
import tempfile
import unicodedata

import numpy as np
import pandas as pd
import polars as pl
from IPython.display import display

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "data" / "games.csv").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Cannot find data/games.csv from the current directory or its parents.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from steam_data import OWNER_COLUMNS, OWNER_LB, OWNER_UB, OWNER_MID, add_estimated_owner_columns, ownership_target

DATA_PATH = PROJECT_ROOT / "data"
CSV_PATH = DATA_PATH / "games.csv"
OUTPUT_PATH = DATA_PATH / "games-extra.parquet"
MANIFEST_PATH = OUTPUT_PATH.with_suffix(".manifest.json")
CATALOG_PATH = OUTPUT_PATH.with_suffix(".feature-catalog.csv")
CLEANING_PATH = OUTPUT_PATH.with_suffix(".cleaning-report.csv")
DATASET_VERSION = "steam-games-extra-v1"
OBSERVATION_DATE = pd.Timestamp("2023-07-05")
CLASSES = [0, 1, 2]

# Compatibility option only. Leave False for model experiments with fold-fitted vocabularies.
INCLUDE_LEGACY_ONE_HOT = False
print("Source:", CSV_PATH)
print("Destination:", OUTPUT_PATH)

Source: /mnt/c/workspace3/machine-learning-engineer-amsterdam (29)/data/games.csv
Destination: /mnt/c/workspace3/machine-learning-engineer-amsterdam (29)/data/games-extra.parquet


## 2. Load and clean the data

These are the same fixed cleaning rules as the experiment notebook: remove exact duplicate rows, validate AppIDs, normalize numeric/platform values, clean descriptions and genre/tag labels, parse dates and ownership intervals, and apply the eligibility filters. Conflicting rows with the same AppID or intervals spanning a target threshold raise an error rather than silently assigning a class.

Missing numeric values remain missing. Medians, scaling and any learned outlier thresholds must be fitted on training folds, not on the complete exported dataset. Cleaning action counts may overlap and should not be added together.

In [2]:
raw = pd.read_csv(CSV_PATH, low_memory=False)
required = ["AppID", "Release date", "Estimated owners", "Price", "Required age",
            "Windows", "Mac", "Linux", "Achievements", "Genres", "Tags", "About the game"]
cleaning_actions = []
def log_cleaning(action, affected_rows):
    cleaning_actions.append({"action": action, "affected_rows": int(affected_rows)})

def normalize_labels(value):
    if pd.isna(value):
        return ""
    tokens = [" ".join(unicodedata.normalize("NFKC", item).split()).casefold()
              for item in str(value).split(",")]
    return ",".join(dict.fromkeys(token for token in tokens if token))

def clean_description(value):
    if pd.isna(value):
        return ""
    text = html.unescape(unicodedata.normalize("NFKC", str(value)))
    text = re.sub(r"<[^>]+>", " ", text)
    return " ".join(text.split())

def clean_game_data(source):
    cleaned = source.copy()
    cleaned.columns = cleaned.columns.str.strip()
    assert cleaned.columns.is_unique, "Column normalization created duplicate names."
    assert set(required) <= set(cleaned.columns), f"Missing columns: {set(required) - set(cleaned.columns)}"
    duplicates = cleaned.duplicated()
    log_cleaning("Exact duplicate rows removed", duplicates.sum())
    cleaned = cleaned.loc[~duplicates].copy()
    appid = pd.to_numeric(cleaned["AppID"], errors="coerce")
    valid_id = np.isfinite(appid) & appid.gt(0) & appid.mod(1).eq(0)
    log_cleaning("Invalid AppIDs excluded", (~valid_id).sum())
    cleaned = cleaned.loc[valid_id].copy()
    cleaned["AppID"] = appid.loc[valid_id].astype("int64")
    if cleaned["AppID"].duplicated().any():
        raise ValueError("Conflicting rows share an AppID; resolve them before splitting.")

    for column in ["Price", "Required age", "Achievements"]:
        original = cleaned[column]
        numeric = pd.to_numeric(original, errors="coerce")
        invalid = ~np.isfinite(numeric) | numeric.lt(0)
        log_cleaning(f"{column}: invalid nonmissing values set to missing", (invalid & original.notna()).sum())
        log_cleaning(f"{column}: originally missing", original.isna().sum())
        cleaned[column] = numeric.mask(invalid)
    for column in ["Windows", "Mac", "Linux"]:
        original = cleaned[column]
        mapped = original.astype("string").str.strip().str.casefold().map(
            {"true": 1., "false": 0., "1": 1., "0": 0., "1.0": 1., "0.0": 0., "yes": 1., "no": 0.})
        log_cleaning(f"{column}: unknown nonmissing flags set to missing", (original.notna() & mapped.isna()).sum())
        cleaned[column] = mapped
    for column in ["Genres", "Tags"]:
        original = cleaned[column].fillna("").astype(str)
        cleaned[column] = cleaned[column].map(normalize_labels)
        log_cleaning(f"{column}: normalized or deduplicated", original.ne(cleaned[column]).sum())
    original = cleaned["About the game"].fillna("").astype(str)
    cleaned["About the game"] = cleaned["About the game"].map(clean_description)
    log_cleaning("Descriptions: HTML/Unicode/whitespace normalized", original.ne(cleaned["About the game"]).sum())

    release_string = cleaned["Release date"].astype("string").str.strip()
    release_string = release_string.str.replace(r"^([A-Za-z]{3}) (\d{4})$", r"\1 1, \2", regex=True)
    cleaned["release_date"] = pd.to_datetime(release_string, format="mixed", errors="coerce")
    # Define lower bound, upper bound and midpoint from the source range BEFORE use.
    cleaned = add_estimated_owner_columns(cleaned, strict=False)
    valid_dates = cleaned["release_date"].notna()
    valid_bounds = cleaned[OWNER_COLUMNS].notna().all(axis=1)
    log_cleaning("Invalid release dates excluded", (~valid_dates).sum())
    log_cleaning("Invalid ownership intervals excluded", (~valid_bounds).sum())
    cleaned = cleaned.loc[valid_dates & valid_bounds].copy()
    for edge in [20000, 100000]:
        if ((cleaned["Estimated owners (lb)"] < edge) & (cleaned["Estimated owners (ub)"] > edge)).any():
            raise ValueError(f"Ownership interval crosses {edge}; review target definition.")
    positive = cleaned[OWNER_MID] > 0
    old_enough = cleaned["release_date"] <= OBSERVATION_DATE - pd.DateOffset(years=1)
    log_cleaning("Zero-ownership rows excluded", (~positive).sum())
    log_cleaning("Games less than one year old excluded", (~old_enough).sum())
    cleaned = cleaned.loc[positive & old_enough].copy()
    cleaned["target"] = ownership_target(cleaned)
    return cleaned.sort_values("AppID").reset_index(drop=True)

In [3]:
frame = clean_game_data(raw)
cleaning_report = pd.DataFrame(cleaning_actions)
assert frame["AppID"].is_unique and set(frame["target"]) == set(CLASSES)
print(f"Raw games: {len(raw):,}; eligible games: {len(frame):,}; removed: {len(raw) - len(frame):,}")
display(cleaning_report)
display(frame[["Estimated owners", *OWNER_COLUMNS, "target"]].drop_duplicates().sort_values(OWNER_LB))

Raw games: 85,103; eligible games: 58,614; removed: 26,489
                                               action  affected_rows
0                        Exact duplicate rows removed              0
1                             Invalid AppIDs excluded              0
2     Price: invalid nonmissing values set to missing              0
3                           Price: originally missing              0
4   Required age: invalid nonmissing values set to...              0
5                    Required age: originally missing              0
6   Achievements: invalid nonmissing values set to...              0
7                    Achievements: originally missing              0
8    Windows: unknown nonmissing flags set to missing              0
9        Mac: unknown nonmissing flags set to missing              0
10     Linux: unknown nonmissing flags set to missing              0
11                 Genres: normalized or deduplicated          81548
12                   Tags: normalized or ded

## 3. Reproduce the experiment features

### What is new compared with `mle-process-data.ipynb`?

The original notebook already contains the source price, required age, platform flags and achievements; it also derives `year` (game age), ownership bounds/midpoint, `Steam score`, the target, and genre/tag indicators. Those are retained or represented compatibly here, rather than counted as new engineered features. The extended notebook adds **17 experiment features** below and **13 additional metadata candidates** in section 4.

These rationales are modeling hypotheses, not measured improvements. Some features overlap with existing inputs; compare feature groups using the same five-fold development splits before deciding which to keep. All formulas here are fixed and label-independent. Fit imputation, scaling, vocabularies and TF-IDF inside training folds.

### Reasons for the 17 experiment features

| Added feature | Definition used in this notebook | Reason for adding it |
| --- | --- | --- |
| `log_price` | `log1p(Price)` | Compress the long right tail of prices so expensive outliers have less influence on distance-based models; express relative price differences more smoothly. This does not replace fold-fitted scaling for KNN. |
| `log_achievements` | `log1p(Achievements)` | Reduce the influence of games with exceptionally many achievements while retaining the distinction between zero and small positive counts. |
| `log_age` | `log1p(year)` | Represent a possible diminishing association between time on the market and accumulated ownership: an extra year may matter more for a young game than an old one. |
| `platform_count` | Sum of Windows, Mac and Linux availability; missing if any flag is unknown | Summarize platform breadth as a possible indicator of audience accessibility, complementing the individual platform flags. It counts these three platforms only. |
| `is_free` | 1 when price is zero, 0 when positive; missing when price is unknown | Expose the distinct free-to-play pricing regime, whose relationship with ownership may differ from price changes among paid games. It is based on price, not the genre/tag label. |
| `age_restricted` | 1 when required age is positive, 0 when zero; missing when unknown | Separate any declared age restriction from no declared restriction, as a simple possible audience-access signal without relying on the exact numeric age. |
| `release_month_sin` | `sin(2π × release_month / 12)` | Represent annual release seasonality on a circular scale; used with cosine so December and January are close in the two-dimensional encoding. |
| `release_month_cos` | `cos(2π × release_month / 12)` | Complete the circular month representation and distinguish months that share the same sine value. Both components are needed to locate the month. |
| `genre_count` | Number of cleaned genre labels, excluding `free to play` | Summarize genre breadth independently of which specific genres occur. Excluding the free-to-play label avoids treating pricing as an additional genre in this count. |
| `category_count` | Number of entries in `Categories` | Approximate the breadth of listed Steam capabilities, such as multiplayer or achievements. It complements genre content; the parser counts entries rather than assessing capability quality. |
| `language_count` | Number of entries in `Supported languages` | Approximate localization breadth and potential audience reach. Counts reflect source entries and parsing, not measured player accessibility or language proficiency. |
| `description_length` | Character count of the cleaned description | Capture listing detail or completeness without learning a text vocabulary. Length is a metadata proxy, not a measure of writing quality. |
| `description_word_count` | Whitespace-separated word count of the cleaned description | Provide another measure of description detail that is less directly tied to character length; test whether it adds value beyond character count. |
| `price_missing` | 1 when the cleaned price is missing | Preserve the fact that price was unavailable or invalid after later imputation, so an imputed price is distinguishable from a genuinely observed value. |
| `age_missing` | 1 when cleaned `Required age` is missing | Distinguish an unknown age restriction from a known zero or an imputed value. This refers to required player age, not game age. |
| `achievements_missing` | 1 when the cleaned achievement count is missing | Distinguish unavailable achievement metadata from a genuine count of zero after imputation. |
| `platform_flags_missing` | Number of missing flags among Windows, Mac and Linux (0–3) | Quantify platform-metadata incompleteness and distinguish an unknown platform count from a known lack of availability. This is a count, not a binary flag. |

Missing numeric values remain missing in this export. Unknown price/age is not treated as a known free/unrestricted game. A feature that is constant on the training data, such as a missingness flag with no missing cases, cannot discriminate classes in that training set.

These formulas match the shared experiment/serving feature definitions. Exporting them makes the deterministic columns available without repeating feature engineering during exploration.


In [4]:
# Fixed, label-independent transformations. Learned transformations come later in pipelines.
def split_labels(value):
    return [item.strip() for item in str(value).split(",")
            if item.strip() and item.strip().casefold() != "free to play"]

def list_length(value):
    if pd.isna(value) or not str(value).strip():
        return 0
    try:
        parsed = ast.literal_eval(str(value))
        if isinstance(parsed, (list, tuple)):
            return len(parsed)
    except (ValueError, SyntaxError):
        pass
    return len([x for x in str(value).split(",") if x.strip()])

BASE_NUMERIC = ["Price", "Required age", "Windows", "Mac", "Linux", "Achievements", "year"]
ENGINEERED = ["log_price", "log_achievements", "log_age", "platform_count",
              "is_free", "age_restricted", "release_month_sin", "release_month_cos",
              "genre_count", "category_count", "language_count", "description_length",
              "description_word_count", "price_missing", "age_missing",
              "achievements_missing", "platform_flags_missing"]
X = pd.DataFrame(index=frame.index)
for column in ["Price", "Required age", "Achievements"]:
    X[column] = pd.to_numeric(frame[column], errors="coerce")
for column in ["Windows", "Mac", "Linux"]:
    X[column] = frame[column]  # already normalized to 0/1/missing by cleaning
X["year"] = (OBSERVATION_DATE - frame["release_date"]).dt.days / 365.0
for column in ["Genres", "Tags", "About the game"]:
    X[column] = frame[column].fillna("").astype(str)
for source, dest in [("Price", "log_price"), ("Achievements", "log_achievements"), ("year", "log_age")]:
    X[dest] = np.log1p(X[source].clip(lower=0))
X["platform_count"] = X[["Windows", "Mac", "Linux"]].sum(axis=1, min_count=3)
X["is_free"] = X["Price"].eq(0).astype(float).where(X["Price"].notna())
X["age_restricted"] = X["Required age"].gt(0).astype(float).where(X["Required age"].notna())
month = frame["release_date"].dt.month
X["release_month_sin"] = np.sin(2 * np.pi * month / 12)
X["release_month_cos"] = np.cos(2 * np.pi * month / 12)
X["genre_count"] = X["Genres"].map(lambda value: len(split_labels(value)))
for source, dest in [("Categories", "category_count"), ("Supported languages", "language_count")]:
    X[dest] = frame[source].map(list_length) if source in frame else 0
X["description_length"] = X["About the game"].str.len()
X["description_word_count"] = X["About the game"].str.split().str.len()
for source, dest in [("Price", "price_missing"), ("Required age", "age_missing"),
                     ("Achievements", "achievements_missing")]:
    X[dest] = X[source].isna().astype(int)
X["platform_flags_missing"] = X[["Windows", "Mac", "Linux"]].isna().sum(axis=1)
X = X.replace([np.inf, -np.inf], np.nan)
y = frame["target"].copy()
assert X.columns.is_unique and y.index.equals(X.index)
assert X[BASE_NUMERIC + ENGINEERED].select_dtypes(include="number").shape[1] == len(BASE_NUMERIC + ENGINEERED)
assert not set(["target", "AppID", "Estimated owners", *OWNER_COLUMNS, "Positive", "Negative", "Peak CCU"]) & set(X.columns)

## 4. Extend the original processed schema and add metadata candidates

### Reasons for the 13 additional candidates

These columns extend the 17 experiment features above. They are available in `games-extra.parquet` for a separate feature ablation; they are **not automatically included in the current experiment or production model**.

| Added feature | Definition used in this notebook | Reason for adding it |
| --- | --- | --- |
| `release_year` | Calendar year of release | Allow models to represent release-cohort differences, such as changes in market conditions or catalog composition. It overlaps strongly with game age at a fixed observation date; its value must be tested rather than assumed. |
| `release_month` | Calendar month, 1–12 | Offer tree models a direct month value for seasonal splits, as an alternative or complement to the sine/cosine encoding. Numeric month alone does not represent December–January proximity. |
| `release_quarter` | Calendar quarter, 1–4 | Provide a coarser seasonal grouping that may be easier to estimate than individual month effects. It is derived from month and adds a representation rather than new source information. |
| `release_dayofweek` | Release weekday, Monday = 0 through Sunday = 6 | Explore whether release scheduling is associated with ownership. Treat this as categorical or cyclic when appropriate; a larger number does not imply a larger effect. Month-only source dates are assigned day 1 during parsing, so their weekdays are artificial. |
| `tag_count` | Number of cleaned tag labels, excluding `free to play` | Summarize tag coverage or descriptive breadth beyond individual tag identities. Community tagging may grow with popularity, so availability at the prediction time needs checking. |
| `developer_count` | Number of entries in the developer list | Explore whether credited development collaboration is associated with ownership. This counts listed organizations; it does not measure team size, reputation or budget. |
| `publisher_count` | Number of entries in the publisher list | Explore whether the breadth of credited publishing relationships contains useful information. It is not a measure of publisher quality or distribution strength. |
| `full_audio_language_count` | Number of entries in the full-audio-language list | Distinguish voiced localization coverage from general supported-language coverage, which may include interface or subtitle support. |
| `description_missing` | 1 when the cleaned description is empty | Explicitly mark missing listing text rather than asking the model to infer missingness from zero character/word counts or an empty text vector. |
| `genres_missing` | 1 when the parsed genre list is empty | Distinguish unavailable genre metadata from a populated list whose labels may be unseen by a fold-fitted encoder. |
| `tags_missing` | 1 when the parsed tag list is empty | Distinguish absent tags from nonempty tags outside an encoder's learned vocabulary; it complements tag count and tag encoding. |
| `categories_missing` | 1 when the parsed category list is empty | Mark absent category metadata explicitly when interpreting `category_count = 0`; source absence does not prove the game has no capabilities. |
| `languages_missing` | 1 when the supported-language list is empty | Distinguish absent localization metadata from a meaningful observed language list. A zero count does not prove the game supports no languages. |

Empty lists can represent missing source fields or absent optional columns. Their missingness flags describe the exported metadata, not verified real-world absence. Counts use the parser shown below; they are not validated counts of unique organizations or languages. If no missing cases occur, a missingness flag contributes no training signal.

### Supporting schema fields and retained information

Keep every source column and preserve both experiment strings (`Genres`, `Tags`, `About the game`) and the original notebook's lowercase lists (`developers`, `publishers`, `categories`, `genres`, `tags`). The following fields support usability and traceability; they are not automatically selected model inputs.

| Field or representation | Reason for retaining or adding it |
| --- | --- |
| `steam_appid` alongside `AppID` | Preserve compatibility with the original processed dataset and a stable key for split tracking and joins. IDs are excluded from model inputs. |
| `release_date` | Provide a timestamp for pandas date-feature calculations; the original `Release date` remains a Parquet date for compatibility. |
| `Release date (raw)` | Retain the source date text for auditing parse assumptions, including month-only dates. |
| `supported_languages` | Expose the parsed language list for inspection and future encoders rather than requiring repeated parsing of the source string. |
| `full_audio_languages` | Expose parsed audio-localization metadata and support `full_audio_language_count`. |
| Cleaned genre/tag/description strings | Supply normalized inputs for vocabularies and TF-IDF learned inside training folds, instead of relying on full-dataset one-hot encoding. These represent existing source information. |

`Steam score`, ownership bounds/midpoint and target are already derived in the original notebook. They are not new predictor features: ownership fields define the outcome, while Steam score uses post-release reviews and remains analysis-only. No-review games have null Steam score rather than a fabricated zero.

All added predictors are metadata-based hypotheses. Current snapshot values for price, tags, localization or listing text may differ from launch-time values; verify that the same information would exist when a real prediction is requested. Do not infer causal effects or claim performance gains from feature creation alone. Using the 13 additional candidates in production requires validation, an updated serving transformer and a new feature-schema version.


In [5]:
def source_list(value):
    if isinstance(value, (list, tuple, np.ndarray)):
        return [str(item).strip() for item in value if str(item).strip()]
    if pd.isna(value) or not str(value).strip():
        return []
    text = str(value).strip()
    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, (list, tuple)):
            return [str(item).strip() for item in parsed if str(item).strip()]
    except (ValueError, SyntaxError):
        pass
    return [item.strip() for item in text.split(",") if item.strip()]

extra = frame.copy()
# Retain all experiment-ready feature values with the exact same formulas.
for column in X.columns:
    extra[column] = X[column]
extra["steam_appid"] = extra["AppID"]
extra["Release date (raw)"] = extra["Release date"]
extra["Release date"] = extra["release_date"]
for source_column, list_column in [("Developers", "developers"), ("Publishers", "publishers"),
                                  ("Categories", "categories"), ("Genres", "genres"), ("Tags", "tags"),
                                  ("Supported languages", "supported_languages"),
                                  ("Full audio languages", "full_audio_languages")]:
    extra[list_column] = extra[source_column].map(source_list) if source_column in extra else [[] for _ in range(len(extra))]

EXTRA_FEATURES = ["release_year", "release_month", "release_quarter", "release_dayofweek",
                  "tag_count", "developer_count", "publisher_count", "full_audio_language_count",
                  "description_missing", "genres_missing", "tags_missing", "categories_missing",
                  "languages_missing"]
extra["release_year"] = extra["release_date"].dt.year
extra["release_month"] = extra["release_date"].dt.month
extra["release_quarter"] = extra["release_date"].dt.quarter
extra["release_dayofweek"] = extra["release_date"].dt.dayofweek  # Monday=0
extra["tag_count"] = extra["Tags"].map(lambda value: len(split_labels(value)))
for column in ["developers", "publishers", "full_audio_languages"]:
    extra[{"developers": "developer_count", "publishers": "publisher_count",
           "full_audio_languages": "full_audio_language_count"}[column]] = extra[column].map(len)
extra["description_missing"] = extra["About the game"].str.strip().eq("").astype(int)
extra["genres_missing"] = extra["genres"].map(len).eq(0).astype(int)
extra["tags_missing"] = extra["tags"].map(len).eq(0).astype(int)
extra["categories_missing"] = extra["categories"].map(len).eq(0).astype(int)
extra["languages_missing"] = extra["supported_languages"].map(len).eq(0).astype(int)
positive = pd.to_numeric(extra["Positive"], errors="coerce")
negative = pd.to_numeric(extra["Negative"], errors="coerce")
extra["Steam score"] = positive / (positive + negative).where((positive + negative) > 0)

LEGACY_ENCODED_COLUMNS = []
if INCLUDE_LEGACY_ONE_HOT:
    from sklearn.preprocessing import MultiLabelBinarizer
    for column in ["genres", "tags"]:
        encoder = MultiLabelBinarizer()
        encoded = encoder.fit_transform(extra[column])
        names = [f"{column[:-1]}_{label}" for label in encoder.classes_]
        extra = pd.concat([extra, pd.DataFrame(encoded, index=extra.index, columns=names)], axis=1)
        LEGACY_ENCODED_COLUMNS.extend(names)
    print("Legacy indicators use the full dataset vocabulary: analysis/compatibility only; exclude from CV inputs.")
extra = extra.sort_values("steam_appid").reset_index(drop=True)
display(extra[["steam_appid", "Name", *ENGINEERED, *EXTRA_FEATURES]].head(5))

   steam_appid  ...  languages_missing
0           10  ...                  0
1           20  ...                  0
2           30  ...                  0
3           40  ...                  0
4           50  ...                  0

[5 rows x 32 columns]


<extra preprocessing cell 9>:35: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.


## 5. Define the feature catalog and verify data integrity

The exported catalog labels each column as an existing model feature, an additional candidate, raw model text, an outcome, analysis-only or other metadata. Never select all numeric columns automatically: that would include the target, ownership bounds and engagement outcomes.

By default no tag/genre vocabulary or TF-IDF is learned during export. `INCLUDE_LEGACY_ONE_HOT=True` optionally retains the old notebook's globally encoded indicator columns for analysis/backward compatibility. They are explicitly excluded from recommended model inputs. Imputers, scaling, binary vocabularies and TF-IDF belong inside each of the **5 training folds**.

In [6]:
RAW_MODEL_COLUMNS = ["Genres", "Tags", "About the game"]
EXISTING_MODEL_COLUMNS = BASE_NUMERIC + ENGINEERED + RAW_MODEL_COLUMNS
CANDIDATE_MODEL_COLUMNS = EXISTING_MODEL_COLUMNS + EXTRA_FEATURES
OUTCOME_COLUMNS = ["Estimated owners", *OWNER_COLUMNS, "target"]
ANALYSIS_ONLY = ["Positive", "Negative", "Peak CCU", "Metacritic score", "User score", "Score rank",
                 "Recommendations", "Reviews", "Steam score", "Average playtime forever",
                 "Average playtime two weeks", "Median playtime forever", "Median playtime two weeks"]

def feature_role(column):
    if column in OUTCOME_COLUMNS:
        return "outcome_do_not_use_as_input"
    if column in ANALYSIS_ONLY:
        return "analysis_only_post_release"
    if column in LEGACY_ENCODED_COLUMNS:
        return "global_vocabulary_analysis_only"
    if column in RAW_MODEL_COLUMNS:
        return "raw_model_text_fit_encoding_in_folds"
    if column in BASE_NUMERIC:
        return "existing_base_numeric"
    if column in ENGINEERED:
        return "existing_experiment_engineered"
    if column in EXTRA_FEATURES:
        return "additional_metadata_candidate"
    return "retained_metadata_not_selected"

feature_catalog = pd.DataFrame({"column": extra.columns,
    "role": [feature_role(column) for column in extra.columns],
    "dtype_before_export": [str(extra[column].dtype) for column in extra.columns],
    "missing_rows": [int(extra[column].isna().sum()) for column in extra.columns]})
assert extra["steam_appid"].is_unique and extra["steam_appid"].is_monotonic_increasing
assert not set(OUTCOME_COLUMNS + ANALYSIS_ONLY + LEGACY_ENCODED_COLUMNS) & set(CANDIDATE_MODEL_COLUMNS)
assert set(CANDIDATE_MODEL_COLUMNS) <= set(extra)
assert extra["target"].equals(ownership_target(extra))
assert (extra[OWNER_MID] > 0).all()
assert np.allclose(extra[OWNER_MID], (extra[OWNER_LB] + extra[OWNER_UB]) / 2)
assert (extra["release_date"] <= OBSERVATION_DATE - pd.DateOffset(years=1)).all()
assert not np.isinf(extra[BASE_NUMERIC + ENGINEERED + EXTRA_FEATURES].to_numpy(dtype=float)).any()
# Ensure stored experiment features remain identical, including missing-value positions.
pd.testing.assert_frame_equal(extra[BASE_NUMERIC + ENGINEERED + RAW_MODEL_COLUMNS],
                              X[BASE_NUMERIC + ENGINEERED + RAW_MODEL_COLUMNS].reset_index(drop=True), check_dtype=False)
display(feature_catalog.loc[feature_catalog["role"].str.startswith(("existing_", "additional_"))])
print(f"Ready to export: {len(extra):,} rows, {len(extra.columns):,} columns")

                       column  ...  missing_rows
5                Required age  ...             0
6                       Price  ...             0
16                    Windows  ...             0
17                        Mac  ...             0
18                      Linux  ...             0
25               Achievements  ...             0
44                       year  ...             0
45                  log_price  ...             0
46           log_achievements  ...             0
47                    log_age  ...             0
48             platform_count  ...             0
49                    is_free  ...             0
50             age_restricted  ...             0
51          release_month_sin  ...             0
52          release_month_cos  ...             0
53                genre_count  ...             0
54             category_count  ...             0
55             language_count  ...             0
56         description_length  ...             0
57     description_w

## 6. Export a versioned Parquet and metadata

Write the new dataset using Zstandard compression, then read it back and verify row IDs, targets, bounds and feature values. Save the observation date, source/output hashes, schema, row counts, dependency versions, feature lists and eligibility policy in a sidecar manifest.

Rerunning replaces `games-extra.parquet` after a successful temporary-file round trip. `games.parquet` is never overwritten. Parquet replacement is atomic on the same filesystem; the manifest and CSV reports are written afterward, so publish the complete file set together for a release.

In [7]:
def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

parquet_df = pl.from_pandas(extra).with_columns(pl.col("Release date").cast(pl.Date))
# Empty list columns must retain a useful element type, not List(Null).
list_columns = ["developers", "publishers", "categories", "genres", "tags",
                "supported_languages", "full_audio_languages"]
parquet_df = parquet_df.with_columns([pl.col(column).cast(pl.List(pl.String)) for column in list_columns])
fd, temporary_name = tempfile.mkstemp(prefix=".games-extra-", suffix=".parquet", dir=DATA_PATH)
os.close(fd)
temporary_path = Path(temporary_name)
try:
    parquet_df.write_parquet(temporary_path, compression="zstd", statistics=True)
    restored = pl.read_parquet(temporary_path)
    assert restored.shape == parquet_df.shape
    assert restored["steam_appid"].to_list() == parquet_df["steam_appid"].to_list()
    assert restored["target"].to_list() == parquet_df["target"].to_list()
    pd.testing.assert_frame_equal(restored.select(OWNER_COLUMNS + BASE_NUMERIC + ENGINEERED + EXTRA_FEATURES).to_pandas(),
                                  parquet_df.select(OWNER_COLUMNS + BASE_NUMERIC + ENGINEERED + EXTRA_FEATURES).to_pandas())
    temporary_path.chmod(0o644)
    os.replace(temporary_path, OUTPUT_PATH)
finally:
    temporary_path.unlink(missing_ok=True)

feature_catalog["parquet_dtype"] = [str(parquet_df.schema[column]) for column in feature_catalog["column"]]
feature_catalog.to_csv(CATALOG_PATH, index=False)
cleaning_report.to_csv(CLEANING_PATH, index=False)
manifest = {
    "dataset_version": DATASET_VERSION,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_file": CSV_PATH.name, "source_sha256": file_sha256(CSV_PATH),
    "output_file": OUTPUT_PATH.name, "output_sha256": file_sha256(OUTPUT_PATH),
    "observation_date": str(OBSERVATION_DATE.date()),
    "raw_rows": len(raw), "retained_rows": len(extra), "column_count": len(extra.columns),
    "ownership_definitions": {OWNER_LB: "lower endpoint", OWNER_UB: "upper endpoint", OWNER_MID: "(lb + ub) / 2"},
    "eligibility": "release_date <= observation_date minus one calendar year; positive ownership midpoint",
    "class_mapping": {"0": "0–<20k owners", "1": "20k–<100k owners", "2": "100k–200m owners"},
    "existing_experiment_features": EXISTING_MODEL_COLUMNS,
    "additional_candidate_features": EXTRA_FEATURES,
    "excluded_outcome_columns": OUTCOME_COLUMNS,
    "analysis_only_columns": [column for column in ANALYSIS_ONLY if column in extra],
    "legacy_global_one_hot_included": INCLUDE_LEGACY_ONE_HOT,
    "legacy_global_one_hot_columns": LEGACY_ENCODED_COLUMNS,
    "learned_transformations": "fit imputers/scalers/vocabularies/TF-IDF inside training CV folds",
    "python_version": platform.python_version(),
    "package_versions": {name: importlib.metadata.version(name) for name in ["numpy", "pandas", "polars", "pyarrow"]},
    "schema": {column: str(dtype) for column, dtype in parquet_df.schema.items()},
}
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
for path in [OUTPUT_PATH, MANIFEST_PATH, CATALOG_PATH, CLEANING_PATH]:
    print(f"Saved {path.name}: {path.stat().st_size / 1024**2:.2f} MiB")
print("Parquet round-trip checks passed.")

Saved games-extra.parquet: 53.54 MiB
Saved games-extra.manifest.json: 0.01 MiB
Saved games-extra.feature-catalog.csv: 0.00 MiB
Saved games-extra.cleaning-report.csv: 0.00 MiB
Parquet round-trip checks passed.


## 7. Load the new dataset for experiments

Read the saved file and select the declared features rather than every numeric column. Keep raw genre/tag/description strings for the existing pipeline; lowercase list columns are also available for alternative models.

The current production transformer supports `existing_experiment_features`. Using additional candidates requires a new evaluated model, an updated shared serving transformer and an incremented feature schema version. Current snapshot metadata does not establish pre-launch forecasting performance.

In [8]:
games_extra = pl.read_parquet(OUTPUT_PATH)
display(games_extra.select(["steam_appid", "Name", "Release date", *OWNER_COLUMNS,
                           "target", "log_price", "platform_count", "tag_count"]).head(5))
print("Target distribution:")
display(games_extra.group_by("target").len().sort("target"))

# Example inputs for the existing experiment pipeline. Imputation/encoding stays in each fold.
training_frame = games_extra.select(EXISTING_MODEL_COLUMNS + ["target"]).to_pandas()
X_extra = training_frame[EXISTING_MODEL_COLUMNS]
y_extra = training_frame["target"]
assert not set(OUTCOME_COLUMNS) & set(X_extra.columns)
print(f"Existing experiment inputs: {X_extra.shape}; target: {y_extra.shape}")
print(f"Additional metadata candidates available for a separate ablation: {len(EXTRA_FEATURES)}")

shape: (5, 10)
┌────────────┬────────────┬───────────┬───────────┬───┬────────┬───────────┬───────────┬───────────┐
│ steam_appi ┆ Name       ┆ Release   ┆ Estimated ┆ … ┆ target ┆ log_price ┆ platform_ ┆ tag_count │
│ d          ┆ ---        ┆ date      ┆ owners    ┆   ┆ ---    ┆ ---       ┆ count     ┆ ---       │
│ ---        ┆ str        ┆ ---       ┆ (lb)      ┆   ┆ i64    ┆ f64       ┆ ---       ┆ i64       │
│ i64        ┆            ┆ date      ┆ ---       ┆   ┆        ┆           ┆ f64       ┆           │
│            ┆            ┆           ┆ f64       ┆   ┆        ┆           ┆           ┆           │
╞════════════╪════════════╪═══════════╪═══════════╪═══╪════════╪═══════════╪═══════════╪═══════════╡
│ 10         ┆ Counter-St ┆ 2000-11-0 ┆ 1e7       ┆ … ┆ 2      ┆ 2.396986  ┆ 3.0       ┆ 20        │
│            ┆ rike       ┆ 1         ┆           ┆   ┆        ┆           ┆           ┆           │
│ 20         ┆ Team       ┆ 1999-04-0 ┆ 5e6       ┆ … ┆ 2      ┆ 1.790091  ┆